In [1]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.colors import HexColor, white
from reportlab.lib.units import mm
from reportlab.lib.utils import simpleSplit
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas
import pandas as pd
import json, os, datetime
pd.set_option('display.max_columns', None)

In [2]:
RESULTS = 'results'
OUTPUT  = 'release/certificate.pdf'
LOGO    = 'images/logo.png'
SIGN    = 'images/signature.png'
RENEWAL = 365    # dias ATÉ A PRÓXIMA CALIBRAÇÃO RECOMENDADA

STRATEGIES = {'kalman_fusion': 'Fusão de Kalman', 'linear_fit': 'Ajuste linear', 'machine_learning': 'Aprendizado de máquina'}
OPTIONS    = json.loads(open('../info.json', 'r').read())
TEST       = OPTIONS.get('test', 1)
RECORD     = f'../Dataset/test{TEST}/results'
strategy   = json.loads(open(f'{RECORD}/attitude.json', 'r').read())
today      = datetime.date.today()
NUMBER     = f'MO-{today:%Y%m%d}-{TEST:04d}'

INFO = {
    'Equipamento':        'Measure MRU',
    'Número de série':    OPTIONS.get('id', '0001'),
    'Data da calibração': f'{today:%d/%m/%Y}',
    'Próxima calibração': f'{today + datetime.timedelta(days=RENEWAL):%d/%m/%Y}',
    'Estratégia':         STRATEGIES[strategy['strategy']],
    'Referência':         'Plataforma de referência',
    'Executado por':      'Measure Offshore'
}

INFO

{'Equipamento': 'Measure MRU',
 'Número de série': 'MIC926199',
 'Data da calibração': '19/11/2026',
 'Próxima calibração': '19/11/2027',
 'Estratégia': 'Fusão de Kalman',
 'Referência': 'Plataforma de referência',
 'Executado por': 'Measure Offshore'}

# REQUISITOS
- Cada linha cobra um limite de uma métrica sobre um grupo de eixos, e o resultado é sempre o pior eixo contra o limite
- Cada eixo ganha uma coluna: roll e pitch na atitude; X, Y e Z nos sensores inerciais, onde o giroscópio e o acelerômetro são uma linha cada
- O `rms_scale_factor` ainda não é calculado pelo `Analysis.ipynb`: a linha do fator de escala só aparece quando ele existir

In [3]:
PLOTTED = ['roll', 'pitch']    # OS DOIS ÂNGULOS QUE O PDF DESENHA
PANEL   = 1.62/3.42           # ALTURA SOBRE LARGURA DOS GRÁFICOS: O PANEL DO Analysis.ipynb

SECTIONS = [
    {
        'title':   'Atitude: roll e pitch',
        'about':   'A plataforma de referência balança o MRU em pitch e em roll, e o ângulo medido é comparado ao dela em repouso e em movimento.',
        'columns': ['Roll', 'Pitch'],
        'tests':   [('Erro RMS em repouso (°)', 'rms_stat', 0.20, PLOTTED), ('Erro RMS em movimento (°)', 'rms_dyn', 0.50, PLOTTED), ('Fator de escala', 'rms_scale_factor', 0.10, PLOTTED)],
        'plots':   True
    },
    {
        'title':   'Sensores inerciais: ruído em repouso',
        'about':   'Desvio padrão de cada eixo com o MRU parado sobre a plataforma, nos trechos de repouso do ensaio.',
        'columns': ['Eixo X', 'Eixo Y', 'Eixo Z'],
        'tests':   [('Giroscópio (°/s)', 'std_stat', 0.30, ['wx', 'wy', 'wz']), ('Acelerômetro (m/s²)', 'std_stat', 0.05, ['ax', 'ay', 'az'])],
        'plots':   False
    }
]


pd.DataFrame(SECTIONS)[['title', 'columns', 'plots']]

,title,columns,plots
0,Atitude: roll e pitch,"[Roll, Pitch]",True
1,Sensores inerciais: ruído em repouso,"[Eixo X, Eixo Y, Eixo Z]",False


# DADOS DO ENSAIO
- O `Analysis.ipynb` deixa em `results/` um `metrics.json` por eixo e, para cada ângulo desenhado, os dois png no tamanho da folha
- Sem os gráficos de roll e de pitch o certificado não sai: eles são o corpo da primeira seção

In [4]:
# NÚMERO COMO SE ESCREVE NO BRASIL
def formatValue(value, digits):
    return f'{value:.{digits}f}'.replace('.', ',')


# O QUE O Analysis.ipynb DEIXOU EM results/ E AS LINHAS DE CADA TABELA COM O PARECER
class ReportData:
    def __init__(self, folder=RESULTS):
        self.folder = folder

    def update(self):
        self.metrics = {}
        self.plots   = {}

        for axis in sorted(os.listdir(self.folder)):
            self.metrics[axis] = json.loads(open(f'{self.folder}/{axis}/metrics.json', 'r').read())

            for name in ['ref_vs_model', 'error']:
                path = f'{self.folder}/{axis}/{name}.png'

                if os.path.exists(path):
                    self.plots[f'{axis}_{name}'] = path

        missing = [axis for axis in PLOTTED if f'{axis}_error' not in self.plots]

        if missing:
            raise FileNotFoundError(f'faltam os gráficos de {missing} em {self.folder}: rode o Analysis.ipynb')

    # UMA LINHA POR REQUISITO COM O VALOR DE CADA EIXO; O PIOR EIXO DECIDE; REQUISITO QUE NENHUM EIXO MEDIU FICA DE FORA
    def getRows(self, section):
        rows = []

        for label, key, limit, axes in section['tests']:
            values = [self.metrics.get(axis, {}).get(key) for axis in axes]
            known  = [value for value in values if value is not None]

            if not known:
                continue

            rows.append({'label': label, 'limit': f'≤ {formatValue(limit, 2)}', 'values': ['—' if value is None else formatValue(value, 4) for value in values], 'passed': max(known) <= limit})

        return rows

    def getVerdict(self):
        rows = [row for section in SECTIONS for row in self.getRows(section)]
        return {'key': 'approved' if all(row['passed'] for row in rows) else 'failed', 'passed': sum(row['passed'] for row in rows), 'total': len(rows)}

    def info(self):
        return self.metrics


data = ReportData()
data.update()

pd.DataFrame(data.info()).T

,std_stat,std_raw,r2,mae,rmse,rms_stat,rms_dyn
ax,0.006844,0.021818,NaN,NaN,NaN,NaN,NaN
ay,0.020961,0.024072,NaN,NaN,NaN,NaN,NaN
az,0.012046,0.019850,NaN,NaN,NaN,NaN,NaN
heave,0.007123,NaN,-0.433154,0.016539,0.036375,0.007121,0.053978
hr,0.005735,NaN,0.730435,0.029669,0.063002,0.005744,0.094284
pitch,0.100687,NaN,-0.879056,0.503020,0.690865,0.129015,1.026210
roll,0.160858,NaN,0.999207,0.151689,0.218853,0.167573,0.269568
wx,0.223354,0.224447,NaN,NaN,NaN,NaN,NaN
wy,0.112450,0.200162,NaN,NaN,NaN,NaN,NaN
wz,0.236759,0.254930,NaN,NaN,NaN,NaN,NaN


# TABELAS
- O mesmo conteúdo que vai para a folha, antes de virar PDF

In [5]:
for number, section in enumerate(SECTIONS, start=1):
    print(f'{number}. {section["title"]}')
    display(pd.DataFrame([[row['label'], row['limit'], *row['values'], 'Aprovado' if row['passed'] else 'Reprovado'] for row in data.getRows(section)], columns=['Parâmetro', 'Requisito'] + section['columns'] + ['Resultado']))

data.getVerdict()

1. Atitude: roll e pitch


,Parâmetro,Requisito,Roll,Pitch,Resultado
0,Erro RMS em repouso (°),"≤ 0,20","0,1676","0,1290",Aprovado
1,Erro RMS em movimento (°),"≤ 0,50","0,2696","1,0262",Reprovado


2. Sensores inerciais: ruído em repouso


,Parâmetro,Requisito,Eixo X,Eixo Y,Eixo Z,Resultado
0,Giroscópio (°/s),"≤ 0,30","0,2234","0,1125","0,2368",Aprovado
1,Acelerômetro (m/s²),"≤ 0,05","0,0068","0,0210","0,0120",Aprovado


{'key': 'failed', 'passed': 3, 'total': 4}

# DESENHO DA FOLHA
- Uma folha A4 com a cara do `Documents/Datasheet`: a Source Sans 3 embutida (`fonts/`, licença OFL), a paleta roxa e verde-água, títulos numerados em caixa alta com o fio roxo e tabelas de cabeçalho roxo-escuro com linhas alternadas
- De cima para baixo: cabeçalho com o logo, identificação e parecer lado a lado, as seções com a tabela de cada uma, os gráficos de roll e pitch, as observações com a assinatura e o rodapé
- O parecer sai das tabelas: aprovado só com todos os requisitos atendidos
- Todo `draw` recebe o `y` onde começa e devolve o `y` onde parou, que é como a página desce

In [6]:
REGULAR  = 'SourceSans3'
SEMIBOLD = 'SourceSans3-Semibold'
BOLD     = 'SourceSans3-Bold'

for name, file in [(REGULAR, 'Regular'), (SEMIBOLD, 'Semibold'), (BOLD, 'Bold')]:
    pdfmetrics.registerFont(TTFont(name, f'fonts/SourceSans3-{file}.ttf'))

DEEP   = HexColor('#3c0f78')    # A PALETA DO Documents/Datasheet: OS DOCUMENTOS DA MEASURE COM A MESMA CARA
PURPLE = HexColor('#6b39bd')
INK    = HexColor('#1f1a2e')
MUTED  = HexColor('#615b70')
LINE   = HexColor('#ddd6ea')
ZEBRA  = HexColor('#f6f4f9')
TEAL   = HexColor('#2a9d8f')
RED    = HexColor('#b42318')

VERDICTS = {
    'approved': {'title': 'APROVADO', 'mark': 'check', 'label': 'Aprovado', 'color': TEAL, 'ink': HexColor('#1d7468'), 'background': HexColor('#e8f5f2'), 'text': 'O MRU atende a todos os requisitos de calibração deste certificado.'},
    'failed':   {'title': 'REPROVADO', 'mark': 'cross', 'label': 'Reprovado', 'color': RED, 'ink': RED, 'background': HexColor('#fdecea'), 'text': 'O MRU não atende a todos os requisitos: os itens reprovados estão marcados nas tabelas.'}
}

MARKS = {    # AS MARCAS EM TRAÇOS, NA FRAÇÃO DO TAMANHO: O ✓ DA FONTE PARECE UM √
    'check': [[(-0.45, 0.0), (-0.12, -0.34), (0.45, 0.36)]],
    'cross': [[(-0.36, -0.36), (0.36, 0.36)], [(-0.36, 0.36), (0.36, -0.36)]]
}

NOTES = ('Atesta que o MRU foi calibrado conforme os procedimentos internos da Measure Offshore; os resultados valem para o equipamento '
         'identificado, nas condições do ensaio. Erro é o ângulo medido pelo MRU menos o da plataforma de referência, nos trechos em que ela '
         'está parada (repouso) e em balanço (movimento). Recomenda-se nova calibração em 12 meses.')


# A FOLHA DO CERTIFICADO, DE CIMA PARA BAIXO
class PDFReport:
    W, H   = A4
    MARGIN = 16 * mm
    ROW    = 16
    GAP    = 12

    def __init__(self, data, output=OUTPUT):
        self.data   = data
        self.output = output
        self.number = NUMBER
        self.info   = INFO

    def width(self):
        return self.W - 2*self.MARGIN

    def drawText(self, x, y, text, font, size, color, align='left', space=0):
        c = self.canvas
        c.setFont(font, size); c.setFillColor(color)
        {'left': c.drawString, 'right': c.drawRightString, 'center': c.drawCentredString}[align](x, y, text, charSpace=space)

    def drawMark(self, x, y, mark, size, color, width):
        c = self.canvas
        c.saveState()
        c.setStrokeColor(color); c.setLineWidth(width); c.setLineCap(1); c.setLineJoin(1)

        for stroke in MARKS[mark]:
            path = c.beginPath()
            path.moveTo(x + stroke[0][0]*size, y + stroke[0][1]*size)

            for dx, dy in stroke[1:]:
                path.lineTo(x + dx*size, y + dy*size)

            c.drawPath(path, stroke=1, fill=0)

        c.restoreState()

    def drawLines(self, x, y, text, font, size, color, width, leading):
        for line in simpleSplit(text, font, size, width):
            self.drawText(x, y, line, font, size, color)
            y -= leading

        return y

    # O CABEÇALHO DO DATASHEET: LOGO À ESQUERDA, O DOCUMENTO À DIREITA E O FIO ROXO EMBAIXO
    def drawHeader(self):
        c, top = self.canvas, self.H - 13*mm

        c.drawImage(LOGO, self.MARGIN, top - 36, height=42, width=42*262/175, mask='auto')
        self.drawText(self.W - self.MARGIN, top - 4, 'CERTIFICADO DE CALIBRAÇÃO', BOLD, 17, DEEP, 'right', 0.4)
        self.drawText(self.W - self.MARGIN, top - 19, 'Unidade de referência de movimento · Measure MRU', REGULAR, 9, MUTED, 'right')
        self.drawText(self.W - self.MARGIN, top - 33, f'Nº {self.number}', SEMIBOLD, 9, INK, 'right')

        c.setStrokeColor(PURPLE); c.setLineWidth(0.9)
        c.line(self.MARGIN, top - 44, self.W - self.MARGIN, top - 44)
        return top - 44

    # OS DADOS DO ENSAIO EM DUAS COLUNAS À ESQUERDA E O PARECER À DIREITA, NA MESMA ALTURA
    def drawSummary(self, top):
        c       = self.canvas
        height  = 100
        left    = self.width() * 0.6
        right   = self.width() - left - self.GAP
        x       = self.MARGIN + left + self.GAP
        verdict = self.data.getVerdict()
        style   = VERDICTS[verdict['key']]
        items   = list(self.info.items())
        half    = (len(items) + 1) // 2

        self.drawText(self.MARGIN, top - 9, 'IDENTIFICAÇÃO', BOLD, 7.5, DEEP, space=0.5)

        for index, (label, value) in enumerate(items):
            column, row = divmod(index, half)
            px, py      = self.MARGIN + column * left/2, top - 26 - row*22
            self.drawText(px, py, label, REGULAR, 7, MUTED)
            self.drawText(px, py - 10, str(value), SEMIBOLD, 9.5, INK)

        c.setFillColor(style['background'])
        c.roundRect(x, top - height, right, height, 4, fill=1, stroke=0)
        c.setFillColor(style['color'])
        c.rect(x, top - height, 2.5, height, fill=1, stroke=0)

        c.circle(x + 25, top - 27, 12, fill=1, stroke=0)
        self.drawMark(x + 25, top - 27, style['mark'], 11, white, 2.2)
        self.drawText(x + 44, top - 25, style['title'], BOLD, 17, style['ink'], space=0.4)
        self.drawText(x + 44, top - 38, f'{verdict["passed"]} de {verdict["total"]} requisitos atendidos', SEMIBOLD, 8.5, INK)
        self.drawLines(x + 14, top - 60, style['text'], REGULAR, 8.5, INK, right - 26, 11)
        return top - height

    # O TÍTULO NUMERADO DO DATASHEET, EM CAIXA ALTA COM O FIO ROXO, E A DESCRIÇÃO DO ENSAIO
    def drawSection(self, top, number, section):
        c = self.canvas

        c.setFillColor(PURPLE)
        c.rect(self.MARGIN, top - 14, 14, 14, fill=1, stroke=0)
        self.drawText(self.MARGIN + 7, top - 10.5, str(number), BOLD, 9.5, white, 'center')
        self.drawText(self.MARGIN + 21, top - 11, section['title'].upper(), BOLD, 11, INK, space=0.3)

        c.setStrokeColor(PURPLE); c.setLineWidth(0.6)
        c.line(self.MARGIN, top - 19, self.W - self.MARGIN, top - 19)
        return self.drawLines(self.MARGIN, top - 30, section['about'], REGULAR, 8.5, MUTED, self.width(), 11) - 1

    # CABEÇALHO ROXO, LINHAS ALTERNADAS E O RESULTADO DE CADA REQUISITO NUMA ETIQUETA COLORIDA
    def drawTable(self, top, section):
        c       = self.canvas
        rows    = self.data.getRows(section)
        headers = ['Parâmetro', 'Requisito'] + section['columns'] + ['Resultado']
        first   = self.width() * 0.36
        last    = self.width() * 0.16
        middle  = (self.width() - first - last) / (len(headers) - 2)
        widths  = [first] + [middle] * (len(headers) - 2) + [last]
        starts  = [self.MARGIN + sum(widths[:index]) for index in range(len(widths))]
        y       = top

        c.setFillColor(DEEP)
        c.rect(self.MARGIN, y - self.ROW, self.width(), self.ROW, fill=1, stroke=0)

        for index, header in enumerate(headers):
            self.drawText(starts[index] + 6 if index == 0 else starts[index] + widths[index]/2, y - 11, header, SEMIBOLD, 8, white, 'left' if index == 0 else 'center')

        y -= self.ROW

        for line, row in enumerate(rows):
            if line % 2:
                c.setFillColor(ZEBRA)
                c.rect(self.MARGIN, y - self.ROW, self.width(), self.ROW, fill=1, stroke=0)

            cells = [row['limit']] + row['values']
            self.drawText(starts[0] + 6, y - 11, row['label'], REGULAR, 8.5, INK)

            for index, cell in enumerate(cells, start=1):
                self.drawText(starts[index] + widths[index]/2, y - 11, cell, REGULAR, 8.5, MUTED if index == 1 else INK, 'center')

            style = VERDICTS['approved' if row['passed'] else 'failed']
            c.setFillColor(style['background'])
            c.roundRect(starts[-1] + widths[-1]/2 - 31, y - 12.5, 62, 10.5, 5.25, fill=1, stroke=0)
            label = starts[-1] + widths[-1]/2 - (pdfmetrics.stringWidth(style['label'], SEMIBOLD, 7.5) - 9) / 2
            self.drawMark(label - 6, y - 7.6, style['mark'], 6, style['ink'], 1.1)
            self.drawText(label, y - 10.3, style['label'], SEMIBOLD, 7.5, style['ink'])

            c.setStrokeColor(LINE); c.setLineWidth(0.6)
            c.line(self.MARGIN, y - self.ROW, self.W - self.MARGIN, y - self.ROW)
            y -= self.ROW

        return y

    # DETALHE EM MOVIMENTO EM CIMA E ERRO DO ENSAIO INTEIRO EMBAIXO, UM ÂNGULO POR COLUNA, NO TAMANHO EM QUE FORAM DESENHADOS
    def drawPlots(self, top):
        width  = (self.width() - self.GAP) / 2
        height = width * PANEL
        y      = top

        for name in ['ref_vs_model', 'error']:
            for column, axis in enumerate(PLOTTED):
                self.canvas.drawImage(self.data.plots[f'{axis}_{name}'], self.MARGIN + column*(width + self.GAP), y - height, width=width, height=height, mask='auto')

            y -= height + 6

        return y + 6

    # AS OBSERVAÇÕES À ESQUERDA E A ASSINATURA DO RESPONSÁVEL À DIREITA
    def drawClosing(self, top):
        c     = self.canvas
        left  = self.width() * 0.6
        x     = self.MARGIN + left + 2*self.GAP
        width = self.W - self.MARGIN - x

        self.drawText(self.MARGIN, top - 9, 'OBSERVAÇÕES', BOLD, 7.5, DEEP, space=0.5)
        bottom = self.drawLines(self.MARGIN, top - 22, NOTES, REGULAR, 7.5, MUTED, left, 10)

        c.drawImage(SIGN, x + width/2 - 52, top - 44, width=104, height=32.5, mask='auto')
        c.setStrokeColor(INK); c.setLineWidth(0.5)
        c.line(x, top - 46, x + width, top - 46)
        self.drawText(x + width/2, top - 57, 'Eng. de Calibração', SEMIBOLD, 8.5, INK, 'center')
        self.drawText(x + width/2, top - 67, 'Responsável técnico · Measure Offshore', REGULAR, 7.5, MUTED, 'center')
        return min(bottom, top - 67)

    def drawFooter(self):
        c, y = self.canvas, 12*mm

        c.setStrokeColor(LINE); c.setLineWidth(0.6)
        c.line(self.MARGIN, y + 10, self.W - self.MARGIN, y + 10)
        self.drawText(self.MARGIN, y, 'Measure Offshore · measureoffshore.com', SEMIBOLD, 7.5, DEEP)
        self.drawText(self.W/2, y, f'Certificado {self.number} · emitido em {datetime.datetime.now():%d/%m/%Y %H:%M}', REGULAR, 7.5, MUTED, 'center')
        self.drawText(self.W - self.MARGIN, y, 'Página 1 de 1', REGULAR, 7.5, MUTED, 'right')

    def export(self):
        os.makedirs(os.path.dirname(self.output), exist_ok=True)
        self.canvas = canvas.Canvas(self.output, pagesize=A4, initialFontName=REGULAR)
        self.canvas.setTitle(f'Certificado de calibração {self.number}')
        self.canvas.setAuthor('Measure Offshore')
        self.canvas.setSubject(f'Calibração do MRU · {" · ".join(str(value) for value in self.info.values())}')
        self.canvas.setCreator('Calibration/Certificate')

        y = self.drawSummary(self.drawHeader() - 16)

        for number, section in enumerate(SECTIONS, start=1):
            y = self.drawSection(y - 24, number, section)
            y = self.drawTable(y - 4, section)

            if section['plots']:
                y = self.drawPlots(y - 10)

        self.drawClosing(y - 26)
        self.drawFooter()
        self.canvas.save()
        return self.output

# GERANDO O CERTIFICADO

In [7]:
report = PDFReport(data)
print(report.export())

release/certificate.pdf
